In [ ]:
import os
import re
import time
import json
import shutil
import signal
import secrets
import subprocess
import urllib.request
from getpass import getpass
from pathlib import Path

print("Installing Open WebUI and helper packages...")
subprocess.check_call([
    "python", "-m", "pip", "install", "-q",
    "open-webui",
    "requests",
    "nest_asyncio"
])

print("\nEnter your OpenAI API key securely.")
openai_api_key = getpass("OpenAI API Key: ").strip()

if not openai_api_key:
    raise ValueError("OpenAI API key cannot be empty.")

default_model = input("Default model to use inside Open WebUI [gpt-4o-mini]: ").strip()
if not default_model:
    default_model = "gpt-4o-mini"

In [ ]:
os.environ["ENABLE_OPENAI_API"] = "True"
os.environ["OPENAI_API_KEY"] = openai_api_key
os.environ["OPENAI_API_BASE_URL"] = "https://api.openai.com/v1"
os.environ["WEBUI_SECRET_KEY"] = secrets.token_hex(32)
os.environ["WEBUI_NAME"] = "Open WebUI on Colab"
os.environ["DEFAULT_MODELS"] = default_model

data_dir = Path("/content/open-webui-data")
data_dir.mkdir(parents=True, exist_ok=True)
os.environ["DATA_DIR"] = str(data_dir)

In [ ]:
cloudflared_path = Path("/content/cloudflared")
if not cloudflared_path.exists():
    print("\nDownloading cloudflared...")
    url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
    urllib.request.urlretrieve(url, cloudflared_path)
    cloudflared_path.chmod(0o755)

print("\nStarting Open WebUI server...")

server_log = open("/content/open-webui-server.log", "w")
server_proc = subprocess.Popen(
    ["open-webui", "serve"],
    stdout=server_log,
    stderr=subprocess.STDOUT,
    env=os.environ.copy()
)

In [ ]:
local_url = "http://127.0.0.1:8080"
ready = False
for _ in range(120):
    try:
        import requests
        r = requests.get(local_url, timeout=2)
        if r.status_code < 500:
            ready = True
            break
    except Exception:
        pass
    time.sleep(2)

if not ready:
    server_log.close()
    with open("/content/open-webui-server.log", "r") as f:
        logs = f.read()[-4000:]
    raise RuntimeError(
        "Open WebUI did not start successfully.\n\n"
        "Recent logs:\n"
        f"{logs}"
    )

print("Open WebUI is running locally at:", local_url)

print("\nCreating public tunnel...")

tunnel_proc = subprocess.Popen(
    [str(cloudflared_path), "tunnel", "--url", local_url, "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

In [1]:
public_url = None
start_time = time.time()
while time.time() - start_time < 90:
    line = tunnel_proc.stdout.readline()
    if not line:
        time.sleep(1)
        continue
    match = re.search(r"https://[-a-zA-Z0-9]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break

if not public_url:
    with open("/content/open-webui-server.log", "r") as f:
        server_logs = f.read()[-3000:]
    raise RuntimeError(
        "Tunnel started but no public URL was captured.\n\n"
        "Open WebUI server logs:\n"
        f"{server_logs}"
    )

print("\n" + "=" * 80)
print("Open WebUI is ready.")
print("Public URL:", public_url)
print("Local URL :", local_url)
print("=" * 80)

print("\nWhat to do next:")
print("1. Open the Public URL.")
print("2. Create your admin account the first time you open it.")
print("3. Go to the model selector and choose:", default_model)
print("4. Start chatting with OpenAI through Open WebUI.")

print("\nUseful notes:")
print("- Your OpenAI API key was passed through environment variables.")
print("- Data persists only for the current Colab runtime unless you mount Drive.")
print("- If the tunnel stops, rerun the cell.")

def tail_open_webui_logs(lines=80):
    log_path = "/content/open-webui-server.log"
    if not os.path.exists(log_path):
        print("No server log found.")
        return
    with open(log_path, "r") as f:
        content = f.readlines()
    print("".join(content[-lines:]))

def stop_open_webui():
    global server_proc, tunnel_proc, server_log
    for proc in [tunnel_proc, server_proc]:
        try:
            if proc and proc.poll() is None:
                proc.terminate()
        except Exception:
            pass
    try:
        server_log.close()
    except Exception:
        pass
    print("Stopped Open WebUI and tunnel.")

print("\nHelpers available:")
print("- tail_open_webui_logs()")
print("- stop_open_webui()")

Installing Open WebUI and helper packages...

Enter your OpenAI API key securely.
Default model to use inside Open WebUI [gpt-4o-mini]: 


Starting Open WebUI server...
Open WebUI is running locally at: http://127.0.0.1:8080

Creating public tunnel...

Open WebUI is ready.
Public URL: https://mechanics-spirits-tags-injuries.trycloudflare.com
Local URL : http://127.0.0.1:8080

What to do next:
1. Open the Public URL.
2. Create your admin account the first time you open it.
3. Go to the model selector and choose: gpt-4o-mini
4. Start chatting with OpenAI through Open WebUI.

Useful notes:
- Your OpenAI API key was passed through environment variables.
- Data persists only for the current Colab runtime unless you mount Drive.
- If the tunnel stops, rerun the cell.

Helpers available:
- tail_open_webui_logs()
- stop_open_webui()
